In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications import MobileNetV2
from keras import Sequential
from keras.layers import Dense,BatchNormalization,Dropout,GlobalAveragePooling2D
from keras.regularizers import L2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [7]:
test_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\Image_Classifier\test"
train_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\Image_Classifier\train"
validation_path = r"C:\Users\user\OneDrive - REVA University\Projects\PlantHealth_AI\Image_Classifier\Validation"

In [8]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 30

In [9]:
train_datagen = ImageDataGenerator(
    rescale=1 / 255.0,
    horizontal_flip=True,
    rotation_range=10
)

val_datagen = ImageDataGenerator(
    rescale=1 / 255.0
)

test_datagen = ImageDataGenerator(
    rescale=1 / 255.0
)

In [10]:
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)


val_generator = val_datagen.flow_from_directory(
    validation_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


Found 32000 images belonging to 2 classes.
Found 4000 images belonging to 2 classes.
Found 4000 images belonging to 2 classes.


In [11]:
print("Classes:", train_generator.class_indices)

print("Training images:", train_generator.samples)
print("Validation images:", val_generator.samples)
print("Testing images:", test_generator.samples)

Classes: {'Disease': 0, 'Pest': 1}
Training images: 32000
Validation images: 4000
Testing images: 4000


In [12]:
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False,weights="imagenet")
base_model.trainable = False

In [13]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(64,activation="relu"),
    Dropout(0.2),
    Dense(32,activation="relu"),
    Dropout(0.2),
    Dense(18,activation="relu"),
    Dropout(0.2),
    Dense(8,activation="relu"),
    Dropout(0.2),
    Dense(4,activation="relu"),
    Dropout(0.2),
    Dense(train_generator.num_classes, activation="softmax")
])

In [14]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

In [16]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

os.makedirs("backend/Models", exist_ok=True)

model.save("backend/Models/image_classifier_model.keras")

Epoch 1/30
163/500 ━━━━━━━━━━━━━━━━━━━━ 12:50 2s/step - accuracy: 0.8280 - loss: 0.4312

KeyboardInterrupt: 

In [ ]:
predictions = model.predict(test_generator)
test_loss, test_accuracy = model.evaluate(
    test_generator
)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

In [ ]:
y_pred = np.argmax(predictions, axis=1)

y_true = test_generator.classes

class_names = list(test_generator.class_indices.keys())

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))